# Latent-Play: A Playable World Model Learned from Action-Free Video

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rivas-vega/latent-play/blob/refactor/dynamics-lam-overhaul/notebooks/colab_training_and_demo.ipynb)

This notebook executes the end-to-end pipeline of **`latent-play`** on Google Colab Pro (**A100 GPU + High-RAM**) with **automatic checkpoint persistence to Google Drive every 5 minutes**:
- If Colab disconnects at any point, clicking **Run all** will automatically resume training from the latest checkpoint saved in your Google Drive (`MyDrive/latent_play_overhaul/`).
- Datasets (`data/`) are stored on Colab's local NVMe disk for maximum read throughput (`np.memmap`), while `checkpoints/` and `outputs/` are symlinked directly to your Google Drive.

## 1. Setup, Google Drive Mounting, and Repository Cloning

In [ ]:
import os, torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"BF16 Supported: {torch.cuda.is_bf16_supported()}")

In [ ]:
# 1. Clone or update repository in /content/latent-play
import os
if not os.path.exists("/content/latent-play"):
    !git clone -b refactor/dynamics-lam-overhaul https://github.com/rivas-vega/latent-play.git /content/latent-play
%cd /content/latent-play
!git fetch origin refactor/dynamics-lam-overhaul && git checkout refactor/dynamics-lam-overhaul && git pull origin refactor/dynamics-lam-overhaul
os.environ["PYTHONPATH"] = "/content/latent-play"

# 2. Mount Google Drive and symlink checkpoints/ and outputs/ for persistence across disconnections
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/latent_play_overhaul"
    os.makedirs(f"{DRIVE_ROOT}/checkpoints", exist_ok=True)
    os.makedirs(f"{DRIVE_ROOT}/outputs", exist_ok=True)
    for folder in ("checkpoints", "outputs"):
        if os.path.islink(folder):
            os.unlink(folder)
        elif os.path.isdir(folder):
            !cp -rn {folder}/* {DRIVE_ROOT}/{folder}/ 2>/dev/null || true
            !rm -rf {folder}
        os.symlink(f"{DRIVE_ROOT}/{folder}", folder)
    print(f"Checkpoints and outputs symlinked to Google Drive: {DRIVE_ROOT}")
except Exception as e:
    print(f"Running without Google Drive ({e}): saving to local disk.")

# 3. Install dependencies and run the 46 unit tests
!pip install -q -e ".[crafter,dev]" matplotlib
!pytest -q

## 2. Phase A — Quick Validation on `ToyGridEnv` (~3 min on A100)
Verify the entire 4-stage pipeline on `ToyGridEnv` in ~3 minutes before running on Crafter.

In [ ]:
!if [ ! -f data/toy/meta.json ]; then python scripts/collect_data.py --config configs/collect_toy.yaml --total-steps 100000 --workers 4; fi
!python scripts/train_lam.py --config configs/lam_toy.yaml train.amp=auto
!python scripts/train_dyn.py --config configs/dyn_toy.yaml lam_checkpoint=checkpoints/lam_toy/lam_final.pt train.out_dir=checkpoints/dyn_toy_lam train.amp=auto
!python scripts/train_dyn.py --config configs/dyn_toy.yaml train.out_dir=checkpoints/dyn_toy_oracle train.amp=auto
!python scripts/sweep_labels.py --data-root data/toy --lam-ckpt checkpoints/lam_toy/lam_final.pt --dyn-ckpt checkpoints/dyn_toy_lam/dyn_final.pt --oracle-ckpt checkpoints/dyn_toy_oracle/dyn_final.pt --fractions 0.001 0.01 0.05 0.1 1.0 --num-episodes 25 --use-state-context --out-dir outputs/sweep_toy

## 3. Phase B — Data Collection in `Crafter` (300k steps)

In [ ]:
!if [ ! -f data/crafter/meta.json ]; then python scripts/collect_data.py --config configs/collect_crafter.yaml --total-steps 300000 --workers 4; fi
!python scripts/inspect_data.py --data data/crafter --out outputs/crafter_samples.gif

## 4. Train Latent Action Model (LAM) on `Crafter` (`label_fraction=0.0`)
Automatically saves checkpoints every 5 minutes to your Google Drive (`MyDrive/latent_play_overhaul/checkpoints/lam_crafter`).

In [ ]:
!python scripts/train_lam.py --config configs/lam_crafter.yaml train.compile=true train.amp=auto

## 5. Train JEPA Dynamics + SIGReg on `Crafter` (Latent Actions vs. Oracle)
Automatically saves checkpoints every 5 minutes to your Google Drive (`dyn_crafter_lam` and `dyn_crafter_oracle`).

In [ ]:
# 5a. Our model (latent-play): conditioned purely on discovered LAM latent codes
!python scripts/train_dyn.py --config configs/dyn_crafter.yaml lam_checkpoint=checkpoints/lam_crafter/lam_final.pt train.out_dir=checkpoints/dyn_crafter_lam train.amp=auto

# 5b. Action Oracle (Upper Bound): conditioned on 100% ground-truth Crafter actions
!python scripts/train_dyn.py --config configs/dyn_crafter.yaml train.out_dir=checkpoints/dyn_crafter_oracle train.amp=auto

## 6. Label Fraction Sweep (0.1% to 100%), CEM Planning, and VoE Physical Scorer

In [ ]:
!python scripts/sweep_labels.py \
    --use-state-context \
    --data-root data/crafter \
    --lam-ckpt checkpoints/lam_crafter/lam_final.pt \
    --dyn-ckpt checkpoints/dyn_crafter_lam/dyn_final.pt \
    --oracle-ckpt checkpoints/dyn_crafter_oracle/dyn_final.pt \
    --fractions 0.001 0.01 0.05 0.1 1.0 \
    --num-episodes 30 \
    --out-dir outputs/sweep_results

!python scripts/eval_voe.py \
    --config configs/dyn_crafter.yaml \
    --checkpoint checkpoints/dyn_crafter_lam/dyn_final.pt \
    --lam-checkpoint checkpoints/lam_crafter/lam_final.pt \
    --out outputs/voe_metrics.json

!python scripts/plot_paper_figures.py --metrics-json outputs/sweep_results/sweep_metrics.json --out-dir outputs/figures

In [ ]:
!python scripts/eval_planning.py \
    --config configs/dyn_crafter.yaml \
    --checkpoint checkpoints/dyn_crafter_lam/dyn_final.pt \
    --align-ckpt outputs/sweep_results/alignment_f0.01_s1.pt \
    --target-dist 4 \
    --num-episodes 20

from IPython.display import Image
Image("outputs/eval_planning/planned_trajectory.gif")

## 7. Package Results and Logs for Download

In [ ]:
!zip -r latent_play_results.zip outputs/ checkpoints/*/log.jsonl checkpoints/*/confusion.json checkpoints/*/config.json
from google.colab import files
files.download("latent_play_results.zip")